# Reproducing the benchmark on a free Colab T4

This notebook rebuilds the kernel from `main` and re-runs every number quoted in
the README, on the same class of GPU those numbers were measured on.

**Before running:** *Runtime -> Change runtime type -> T4 GPU*. The first cell
checks which GPU you were given; free Colab sometimes hands out an L4 or an A100
instead, and the numbers in the README are T4 numbers.

Then *Runtime -> Run all*. The whole notebook takes a couple of minutes; the
Nsight Compute cell at the end is the slow one and is optional.

In [ ]:
%%bash
nvidia-smi --query-gpu=name,compute_cap,memory.total,driver_version --format=csv,noheader
nvcc --version | tail -2

In [ ]:
%%bash
rm -rf /content/fft
git clone -q https://github.com/matthewd-so/Fast-Fourier-Transform.git /content/fft
cd /content/fft
git log -1 --format='built from %h  %ad  %s' --date=short
SM=$(nvidia-smi --query-gpu=compute_cap --format=csv,noheader | head -1 | tr -d '.')
make bench SM=$SM
make bench_prof SM=$SM

## The headline numbers

N = 2^20, 50 timed iterations, run three times. This is the row the README's
performance table is built from: time, GFLOP/s, speedup over the single-threaded
CPU baseline, and the ratio against cuFFT.

Both accuracy checks print on every run: the relative L2 error against cuFFT's
output, and a forward-then-inverse round trip against the original input.

In [ ]:
%%bash
cd /content/fft
for i in 1 2 3; do ./bench 1048576 50; echo; done

## Why the cuFFT ratio depends on the transform size

The T4's L2 is 4 MB and a complex-float transform is 8N bytes, so the whole
array is resident in L2 up to N = 2^19. Past that every butterfly stage becomes
a full DRAM round trip and the pass-count gap against cuFFT starts to show.

In [ ]:
%%bash
cd /content/fft
for n in 65536 131072 262144 524288 1048576 2097152 4194304; do
    printf 'N=%-9s %-5s MB  ' $n $((n * 8 / 1048576))
    ./bench $n 30 | grep 'of cuFFT' | sed 's/  */ /g'
done

## Per-kernel breakdown (optional, ~1 minute)

Nsight Compute ships with the CUDA toolkit in the Colab image and counter access
is permitted, so `make profile-ncu` works here. This reproduces the table in the
README's *Where the time goes* section.

Kernel replay makes the wall-clock timings printed by `bench` itself meaningless
under the profiler -- ignore them; the per-kernel durations below are the real
measurement. Nsight Systems is not installed in the Colab image, which is why
this uses `profile-ncu` rather than `profile`.

In [ ]:
%%bash
cd /content/fft
export PATH=$PATH:/usr/local/cuda/bin
make profile-ncu N=1048576 NCU_ITERS=1 > /dev/null 2>&1
grep -E 'fft_bitrev_tiled_kernel|fft_shared_kernel|fft_global_kernel|Duration|DRAM Throughput|Memory Throughput +Gbyte|Compute \(SM\) Throughput|Achieved Occupancy' \
    profiles/fft_kernels.txt | sed 's/  */ /g'